In [1]:
from dataclasses import dataclass
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, TimestampType, DoubleType, LongType
from pyspark.storagelevel import StorageLevel
import gc

@dataclass
class Config:
    input_path: str = "/user/data/raw/*.parquet"
    output_path_base: str = "/user/data/kshape/preprocess"
    start_ts: str = "2021-01-01 00:00:00"
    end_ts: str = "2025-12-31 23:59:59"
    bin_minutes: int = 30
    min_duration_minute: float = 1.0
    max_duration_minute: float = 180.0
    min_trip_distance: float = 0.1
    max_trip_distance: float = 50.0
    max_passengers: int = 8
    min_location_id: int = 1
    max_location_id: int = 263
    active_ratio_threshold: float = 0.05
    ewma_alpha: float = 0.3
    ewma_lookback: int = 60

    @property
    def bins_per_day(self):
        return 24 * 60 // self.bin_minutes

    @property
    def bins_per_week(self):
        return self.bins_per_day * 7

    @property
    def required_history_bins(self):
        return max(5, self.bins_per_day, self.ewma_lookback, self.bins_per_week)


class Preprocess:
    def __init__(self, c=Config()):
        self.c = c
        self.spark = (
            SparkSession.builder.appName("Preprocess")
            .config("spark.sql.session.timeZone", "America/New_York")
            .config("spark.sql.files.ignoreCorruptFiles", "true")
            .config("spark.sql.parquet.mergeSchema", "false")
            .config("spark.sql.adaptive.enabled", "true")
            .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
            .config("spark.sql.shuffle.partitions", "400")
            .config("spark.master", "yarn")
            .config("spark.submit.deployMode", "client")
            .config("spark.driver.host", "192.168.1.111")
            .config("spark.driver.bindAddress", "0.0.0.0")
            .getOrCreate()
        )
        self.spark.sparkContext.setLogLevel("ERROR")

        self.schema = StructType([
            StructField("tpep_pickup_datetime", TimestampType(), True),
            StructField("tpep_dropoff_datetime", TimestampType(), True),
            StructField("PULocationID", LongType(), True),
            StructField("DOLocationID", LongType(), True),
            StructField("passenger_count", DoubleType(), True),
            StructField("trip_distance", DoubleType(), True),
            StructField("fare_amount", DoubleType(), True),
            StructField("total_amount", DoubleType(), True),
        ])

    def read(self):
        df = self.spark.read.option("mergeSchema", "false").schema(self.schema).parquet(self.c.input_path)
        df.printSchema()
        print(f"Rows before cleaning: {df.count():,}")
        return df

    def clean(self):
        c = self.c
        df = (
            self.read()
            .withColumn("_source_file", F.input_file_name())
            .select(
                F.col("tpep_pickup_datetime").alias("pickup_dt"),
                F.col("tpep_dropoff_datetime").alias("dropoff_dt"),
                F.col("PULocationID"),
                F.col("DOLocationID"),
                F.col("passenger_count").cast("int"),
                F.col("trip_distance"),
                F.col("fare_amount"),
                F.col("total_amount"),
                F.col("_source_file"),
            )
            .withColumn("trip_duration_minute", (F.unix_timestamp("dropoff_dt") - F.unix_timestamp("pickup_dt")) / 60.0)
            .filter(F.col("PULocationID").between(c.min_location_id, c.max_location_id))
            .filter(F.col("trip_duration_minute").between(c.min_duration_minute, c.max_duration_minute))
            .filter(F.col("trip_distance").isNotNull() & F.col("trip_distance").between(c.min_trip_distance, c.max_trip_distance))
            .filter(F.col("passenger_count").isNull() | F.col("passenger_count").between(1, c.max_passengers))
            .filter(F.col("fare_amount").isNull() | (F.col("fare_amount") >= 0))
            .filter(F.col("total_amount").isNull() | (F.col("total_amount") >= 0))
        )
        print(f"Rows after cleaning: {df.count():,}")
        return df

    def build_panel(self, df):
        c = self.c
        demand = (
            df.withColumn(
                "pickup_bin_30m",
                F.to_timestamp(
                    F.from_unixtime(
                        F.floor(F.unix_timestamp("pickup_dt") / (c.bin_minutes * 60)) * (c.bin_minutes * 60)
                    )
                )
            )
            .groupBy("pickup_bin_30m", "PULocationID")
            .agg(F.count("*").cast("int").alias("pickup_demand"))
        )

        bins = self.spark.sql(f"""
            SELECT explode(
                sequence(
                    to_timestamp('{c.start_ts}'),
                    to_timestamp('{c.end_ts}') - interval {c.bin_minutes} minutes,
                    interval {c.bin_minutes} minutes
                )
            ) AS pickup_bin_30m
        """)

        panel = (
            bins.crossJoin(demand.select("PULocationID").distinct())
            .join(demand, ["pickup_bin_30m", "PULocationID"], "left")
            .fillna({"pickup_demand": 0})
        )

        b = bins.agg(F.min("pickup_bin_30m").alias("min_ts"), F.max("pickup_bin_30m").alias("max_ts")).first()
        secs = int((b["max_ts"] - b["min_ts"]).total_seconds())
        start = b["min_ts"].strftime("%Y-%m-%d %H:%M:%S")
        train_expr = f"timestamp'{start}' + interval {int(secs * 0.70)} seconds"
        val_expr = f"timestamp'{start}' + interval {int(secs * 0.90)} seconds"

        active = (
            panel.filter(F.col("pickup_bin_30m") < F.expr(train_expr))
            .groupBy("PULocationID")
            .agg(
                F.sum(F.when(F.col("pickup_demand") > 0, 1).otherwise(0)).alias("active_bins"),
                F.count("*").alias("bins"),
            )
            .withColumn("active_ratio", F.col("active_bins") / F.col("bins"))
            .filter(F.col("active_ratio") >= c.active_ratio_threshold)
            .select("PULocationID")
        )

        panel = panel.join(active, "PULocationID", "inner").persist(StorageLevel.MEMORY_AND_DISK)
        print(f"Panel rows after active filter: {panel.count():,}")
        return panel, train_expr, val_expr

    def engineer(self, df, train_expr, val_expr):
        c = self.c
        w = Window.partitionBy("PULocationID").orderBy("time_idx")
        w24 = w.rowsBetween(-c.bins_per_day, -1)

        df = (
            df.withColumn("date", F.to_date("pickup_bin_30m"))
            .withColumn("hour", F.hour("pickup_bin_30m").cast("int"))
            .withColumn("minute", F.minute("pickup_bin_30m").cast("int"))
            .withColumn("slot_30m", (F.col("hour") * 2 + F.floor(F.col("minute") / 30)).cast("int"))
            .withColumn("day_of_week", ((F.dayofweek("pickup_bin_30m") + 5) % 7).cast("int"))
            .withColumn("week_of_year", F.weekofyear("pickup_bin_30m").cast("int"))
            .withColumn("month", F.month("pickup_bin_30m").cast("int"))
            .withColumn("day_of_month", F.dayofmonth("pickup_bin_30m").cast("int"))
            .withColumn("is_weekday", F.when(F.col("day_of_week").between(0, 4), 1).otherwise(0).cast("int"))
            .withColumn("is_weekend", F.when(F.col("day_of_week").isin([5, 6]), 1).otherwise(0).cast("int"))
            .withColumn(
                "time_idx",
                ((F.unix_timestamp("pickup_bin_30m") - F.unix_timestamp(F.lit(c.start_ts).cast("timestamp"))) / (c.bin_minutes * 60)).cast("int")
            )
        )

        for i in range(1, 6):
            df = df.withColumn(f"demand_t_{i}", F.lag("pickup_demand", i).over(w))

        df = (
            df.withColumn("rolling_obs_24h", F.count("pickup_demand").over(w24))
            .withColumn("rolling_max_24h", F.max("pickup_demand").over(w24))
            .withColumn("rolling_min_24h", F.min("pickup_demand").over(w24))
            .withColumn("rolling_mean_24h", F.avg("pickup_demand").over(w24))
            .withColumn("rolling_std_24h", F.stddev_samp("pickup_demand").over(w24))
            .withColumn("rolling_skew_24h", F.skewness("pickup_demand").over(w24))
            .withColumn("ha_output", F.lag("pickup_demand", c.bins_per_week).over(w))
        )

        ewma = F.lit(0.0)
        for i in range(1, c.ewma_lookback + 1):
            ewma += (
                F.coalesce(F.lag("pickup_demand", i).over(w), F.lit(0.0)).cast("double")
                * F.lit(c.ewma_alpha * ((1 - c.ewma_alpha) ** (i - 1)))
            )

        df = df.withColumn("ewma_output", ewma.cast("double")).fillna({
            **{f"demand_t_{i}": 0.0 for i in range(1, 6)},
            "rolling_obs_24h": 0,
            "rolling_max_24h": 0.0,
            "rolling_min_24h": 0.0,
            "rolling_mean_24h": 0.0,
            "rolling_std_24h": 0.0,
            "rolling_skew_24h": 0.0,
            "ha_output": 0.0,
            "ewma_output": 0.0,
        })

        df = (
            df.withColumn(
                "dataset_split",
                F.when(F.col("pickup_bin_30m") < F.expr(train_expr), "train")
                .when(F.col("pickup_bin_30m") < F.expr(val_expr), "validation")
                .otherwise("test")
            )
            .withColumn("history_bins_available", F.col("time_idx"))
            .withColumn("enough_history_lag5", (F.col("time_idx") >= 5).cast("int"))
            .withColumn("enough_history_24h", (F.col("time_idx") >= c.bins_per_day).cast("int"))
            .withColumn("enough_history_ewma", (F.col("time_idx") >= c.ewma_lookback).cast("int"))
            .withColumn("enough_history_1w", (F.col("time_idx") >= c.bins_per_week).cast("int"))
            .withColumn("is_cold_start", (F.col("time_idx") < c.required_history_bins).cast("int"))
            .withColumn("ready", (F.col("time_idx") >= c.required_history_bins).cast("int"))
        )

        cols = [
            "pickup_bin_30m", "date", "time_idx", "dataset_split", "PULocationID", "pickup_demand",
            "hour", "minute", "slot_30m", "day_of_week", "week_of_year", "month", "day_of_month", "is_weekday", "is_weekend",
            "demand_t_1", "demand_t_2", "demand_t_3", "demand_t_4", "demand_t_5",
            "rolling_obs_24h", "rolling_max_24h", "rolling_min_24h", "rolling_mean_24h", "rolling_std_24h", "rolling_skew_24h",
            "ha_output", "ewma_output",
            "history_bins_available", "enough_history_lag5", "enough_history_24h", "enough_history_ewma", "enough_history_1w",
            "is_cold_start", "ready",
        ]

        full_panel = df.select(*cols).persist(StorageLevel.MEMORY_AND_DISK)
        model_panel = full_panel.filter(F.col("ready") == 1).persist(StorageLevel.MEMORY_AND_DISK)

        full_panel_count = {r["dataset_split"]: r["count"] for r in full_panel.groupBy("dataset_split").count().collect()}
        model_panel_count = {r["dataset_split"]: r["count"] for r in model_panel.groupBy("dataset_split").count().collect()}

        print("=== Full Panel Counts ===")
        for split, count in full_panel_count.items():
            print(f"  {split}: {count}")

        print("=== Model Panel Counts ===")
        for split, count in model_panel_count.items():
            print(f"  {split}: {count}")

        return full_panel, model_panel

    def cleanup(self, *dfs):
        for df in dfs:
            if df is not None:
                try:
                    df.unpersist(blocking=True)
                except Exception as e:
                    print(f"[WARN] unpersist failed: {e}")

        try:
            self.spark.catalog.clearCache()
        except Exception as e:
            print(f"[WARN] clearCache failed: {e}")

        try:
            gc.collect()
        except Exception as e:
            print(f"[WARN] gc.collect failed: {e}")

    def run(self):
        clean = None
        panel = None
        full_panel = None
        model_panel = None

        try:
            clean = self.clean()
            panel, train_expr, val_expr = self.build_panel(clean)
            full_panel, model_panel = self.engineer(panel, train_expr, val_expr)

            full_panel.write.mode("overwrite").partitionBy("dataset_split").parquet(f"{self.c.output_path_base}/full_panel")
            model_panel.write.mode("overwrite").partitionBy("dataset_split").parquet(f"{self.c.output_path_base}/model_panel")

            print(f"Exported: {self.c.output_path_base}/full_panel")
            print(f"Exported: {self.c.output_path_base}/model_panel")
            print("[SUCCESS] Preprocessing completed.")

        finally:
            self.cleanup(panel, full_panel, model_panel)

            # xóa reference Python
            del clean, panel, full_panel, model_panel

            try:
                gc.collect()
            except Exception as e:
                print(f"[WARN] gc.collect after del failed: {e}")

            try:
                self.spark.stop()
                print("[INFO] Spark stopped.")
            except Exception as e:
                print(f"[WARN] spark.stop failed: {e}")

cfg = Config(
    input_path="/user/data/raw/*.parquet",
    output_path_base="/user/data/kshape/preprocess",
    start_ts="2021-01-01 00:00:00",
    end_ts="2025-12-31 00:00:00",
)

job = Preprocess(cfg)
job.run()

del job
gc.collect()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e86d23bd-5d87-471e-835f-9ce34f94040a;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
downloading https://repos.spark-packages.org/graphframes/graphframes/0.8.3-spark3.5-s_2.12/graphframes-0.8.3-spark3.5-s_2.12.jar ...
	[SUCCESSFUL ] graphframes#graphframes;0.8.3-spark3.5-s_2.12!graphframes.jar (409ms)
downloading https://repo1.maven.org/maven2/org/slf4j/slf4j-api/1.7.16/slf4j-api-1.7.16.jar ...
	[SUCCESSFUL ] org.slf4j#slf4j-api;1.7.16!slf4j-api.jar (99ms)
:: resolution report :: resolve 3197ms :: artifacts dl 514ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	-----------------------

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)



Rows before cleaning: 223,412,046


Rows after cleaning: 92,899,937


Panel rows after active filter: 13,490,400


=== Full Panel Counts ===
  test: 1347808
  train: 9444512
  validation: 2698080
=== Model Panel Counts ===
  test: 1347808
  train: 9392768
  validation: 2698080


Exported: /user/data/kshape/preprocess/full_panel
Exported: /user/data/kshape/preprocess/model_panel
[SUCCESS] Preprocessing completed.
[INFO] Spark stopped.


26